<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">03 Mobile Systems</h1>
<hr>


## **6. Wi-Fi Network Performance**

<div class="alert alert-light">

  **Name:** (add your name here)
  
  **Matriculation Number:** (add your number here)


  Simulations may take a few minutes. If running them is not possible, use the saved plots where available.
  
</div>



### 6.1 Introduction

This notebook examines how Wi-Fi throughput and delay change as more stations share a channel. The Jemula802 simulator models 36 senders transmitting to one receiver. Later experiments change the waiting rules for channel access and the transmission rate of one station.

Offered load is the rate at which traffic enters the system; throughput is the rate successfully delivered. The physical-layer (PHY) data rate describes transmission over the radio and includes no allowance for time spent waiting, acknowledging, or retrying packets. These quantities therefore need to be distinguished when reading the results.

The experiments use a simplified IEEE 802.11 model. Their purpose is to explain contention and airtime sharing, rather than predict the performance of a particular Wi-Fi product.


In [ ]:
import platform

classpath_jemula = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar"
classpath_jemula802 = "./:jemula802/bin:jemula/bin:jemula/lib/xstream-1.3.1.jar:jemula/lib/jfreechart-1.5.3.jar:jemula/lib/jcommon-1.0.23.jar:jemula/lib/commons-io-2.14.0.jar"

if platform.system() == "Windows":
    classpath_jemula   = classpath_jemula.replace(":", ";")
    classpath_jemula802 = classpath_jemula802.replace(":", ";")


#### 6.1.1 Clone or update the simulator


In [ ]:
%%bash
set -e
if [ -d "jemula802/.git" ]; then
  echo "Updating existing repo jemula802 ..."
  git -C jemula802 pull --ff-only
else
  echo "Cloning repo jemula802 ..."
  git clone https://bitbucket.org/lfield/jemula802.git
fi
set -e
if [ -d "jemula/.git" ]; then
  echo "Updating existing repo jemula ..."
  git -C jemula pull --ff-only
else
  echo "Cloning repo jemula ..."
  git clone https://bitbucket.org/lfield/jemula.git
fi

#### 6.1.2 Compile Jemula802 and Jemula


In [ ]:
!javac -g:none -classpath $classpath_jemula -d "jemula/bin" jemula/src/plot/*.java jemula/src/kernel/*.java jemula/src/statistics/*.java
!echo Compilation of jemula done.

!javac -g:none  -classpath $classpath_jemula802 -d "jemula802/bin" jemula802/src/emulator/*.java jemula802/src/layer0_medium/*.java jemula802/src/layer1_802Phy/*.java \
 jemula802/src/layer1_80211Phy/*.java jemula802/src/layer2_802Mac/*.java jemula802/src/layer2_80211Mac/*.java \
 jemula802/src/layer2_802Algorithms/*.java jemula802/src/layer3_network/*.java jemula802/src/layer5_application/*.java \
 jemula802/src/layer4_transport/*.java jemula802/src/station/*.java jemula802/src/visualization/*.java \
 jemula802/src/util/*.java jemula802/src/gui/*.java
!echo Compilation of jemula802 done.

In [ ]:
import os

ROOT_DIR = os.getcwd()
!echo Root directory is $ROOT_DIR

#### 6.1.3 Prepare the evaluation functions


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import os

os.chdir(ROOT_DIR)

In [ ]:
def plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,ylim):
    fig, ax = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax.cla()
    ax.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax.axis([None, None, None, None])
    ax.set_facecolor((.2, 0.2, 0.3))
    plt.xlim(0, np.max(result_total_offer.iloc[:, 0]) / 1000 * 1.05)  # Set x-axis limits
    plt.ylim(0, ylim)  # Set y-axis limits
    plt.xlabel('Simulated Time [s]')
    plt.ylabel('Simulation Results, Mb/s & ms')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks

    plt.plot(result_total_offer.iloc[:, 0] / 1000, result_total_offer.iloc[:, 6], '-', color='g', linewidth=4, label='Offer [Mb/s]')
    plt.plot(result_total_thrp.iloc[:, 0] / 1000, result_total_thrp.iloc[:, 6], '-', color='w', linewidth=1, label='Throughput [Mb/s]')
    plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 3], '-', color='r', linewidth=1, label='Average Delay [ms]')
    #plt.plot(result_total_delay.iloc[:, 0] / 1000, result_total_delay.iloc[:, 5], '-', color='r', linewidth=1)
    plt.legend(loc="upper left", fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

def plot_delay_histogram(aBinSize_ms_string, aHistogram_string):
    fig, ax_distr = plt.subplots(figsize=(8, 4))
    theLineWidth = 2
    ax_distr.cla()
    ax_distr.grid(color="#AAAAAA", linewidth=.5, linestyle='-', alpha=1)
    ax_distr.axis([None, None, None, None])
    ax_distr.set_facecolor((.2, 0.2, 0.3))
    theHistogram = [float(x) for x in aHistogram_string.split()]
    theHistogram = [x/sum(theHistogram) for x in theHistogram]
    theNumOfBins = len(theHistogram)
    theBinSize_ms = float(aBinSize_ms_string)
    theX = [i * theBinSize_ms + theBinSize_ms/2 for i in range(theNumOfBins)] 
    theCumSum = 1-np.cumsum(theHistogram)
    theCumSum = np.insert(theCumSum, 0, 1)
    plt.step([0] + theX, [0] + theHistogram, '-', color='r', linewidth=2, label='PDF')
    plt.step([0] + theX, theCumSum, '-', color='m', linewidth=2, label='CDF')
    plt.grid(True)  # Turn on the grid
    plt.minorticks_on()  # Enable minor ticks
    plt.xlim(0, theNumOfBins * theBinSize_ms * 1.05)  # Set x-axis limits
    plt.xlabel('x [ms]')
    plt.ylabel('PDF: P(delay=x) & CCDF: P(delay>=x)')
    plt.legend(fancybox=True, framealpha=1, facecolor=(.6, 0.61, 0.8))
    plt.tight_layout()

### 6.2 Baseline: increasing the number of senders

The scenario [MobComp_assignment04.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04.xml) defines one central receiver and 36 senders on a ring. One sender starts each second from 2 s to 37 s. Each offers 0.75 Mb/s, bringing the total offered load to 27 Mb/s. The simulation lasts about 50 s.

All senders use the same channel, a 54 Mb/s PHY rate, and traffic with a maximum packet size of 1200 bytes. The configuration sets a 512-byte fragmentation threshold and a 2500-byte Request to Send/Clear to Send (RTS/CTS) threshold. The latter is above the generated packet sizes.

Wi-Fi's Enhanced Distributed Channel Access (EDCA) mechanism assigns channel-access settings to traffic classes, called access categories (ACs). This scenario uses AC2. AIFSN, the Arbitration Inter-Frame Space Number, controls the idle waiting interval before backoff. CWmin and CWmax set the initial and maximum contention-window bounds. A backoff counter is drawn from zero through the current bound. The transmission opportunity (TXOP) limit bounds a transmission opportunity after access is won.

The baseline uses AIFSN = 4, CWmin = 5, and CWmax = 10. Later sections state the changes to this configuration. Results and a copy of the scenario are stored in [results_MobComp_A04](results_MobComp_A04).

<center>
<a href="https://gitlab.ethz.ch/wireless/MobileComputing/-/raw/main/.admin/notebooks/picture/jemula_scenario01.png"><img src="../../.admin/notebooks/picture/jemula_scenario01.png" 
     alt="Scenario - click to open the image" 
     width="80%" 
     style="filter: grayscale(80%);"></a>
</center>

#### 6.2.1 Run the simulation


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO01="jemula802/scenarios/ETH_Course/MobComp_assignment04.xml"
!echo Running scenario $SCENARIO01
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO01

#### 6.2.2 Plot the results

The time-series plot combines offered load and throughput in Mb/s with average delay in ms; the legend identifies the units. The second plot shows the delay distribution. Its descending cumulative curve is the fraction of recorded delays above a threshold, often called the complementary cumulative distribution function (CCDF), although the current legend abbreviates it as CDF.

Observe when throughput stops following offered load and how delay changes as more senders become active.


In [ ]:
if os.path.isdir("results_MobComp_A04"):
    os.chdir("results_MobComp_A04")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay, 180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.3 Increase the waiting interval and maximum backoff

The scenario [MobComp_assignment04_modified_dot11EDCAAIFSN.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCAAIFSN.xml) changes AIFSN from 4 to 8 and CWmax from 10 to 100. CWmin remains 5. Thus, this experiment changes two access parameters, despite the filename mentioning only AIFSN.

A larger AIFSN extends the idle waiting interval; a larger CWmax permits a wider backoff range after failures. Compare throughput and delay with section 6.2 to assess the combined effect. A change in one result cannot be attributed to AIFSN alone.

Results are stored in [results_MobComp_A04_modified_dot11EDCAAIFSN](results_MobComp_A04_modified_dot11EDCAAIFSN).

#### 6.3.1 Select the modified scenario


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO02="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCAAIFSN.xml"
!echo Modified scenario is $SCENARIO02

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO02, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO02, lineterm='')

#for line in diff:
#    print(line)


#### 6.3.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO02

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_dot11EDCAAIFSN"):
    os.chdir("results_MobComp_A04_modified_dot11EDCAAIFSN")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay, 180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.4 Increase the initial contention window

The scenario [MobComp_assignment04_modified_dot11EDCACWmin.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCACWmin.xml) increases CWmin from 5 to 8, while AIFSN remains 4 and CWmax remains 10.

A larger CWmin spreads initial transmission attempts over more backoff choices. This can reduce simultaneous attempts but also increases average waiting time. Compare the results with section 6.2, both before and after many senders become active.

Results are stored in [results_MobComp_A04_modified_dot11EDCACWmin](results_MobComp_A04_modified_dot11EDCACWmin).

#### 6.4.1 Select the modified scenario


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO03="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_dot11EDCACWmin.xml"
!echo Modified scenario is $SCENARIO03

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO03, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO03, lineterm='')

#for line in diff:
#    print(line)

#### 6.4.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO03

In [ ]:
if os.path.isdir("results_MobComp_A04_modified_dot11EDCACWmin"):
    os.chdir("results_MobComp_A04_modified_dot11EDCACWmin")

result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)

plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay ,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

### 6.5 Give station 10 more aggressive access settings

The scenario [MobComp_assignment04_modified_Station10only.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_Station10only.xml) changes station 10 to AIFSN = 2 and CWmin = 1. Other stations retain AIFSN = 4 and CWmin = 5; CWmax remains 10.

These settings let station 10 wait less before attempting transmission. Compare its throughput and delay with station 8 after both flows are active. Then examine total throughput to distinguish an individual advantage from a benefit to the whole network.

Results are stored in [results_MobComp_A04_modified_Station10only](results_MobComp_A04_modified_Station10only).

#### 6.5.1 Select the modified scenario


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO04="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_Station10only.xml"
!echo Modified scenario is $SCENARIO04

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO04, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO04, lineterm='')

#for line in diff:
#    print(line)

#### 6.5.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO04

Results for all traffic:


In [ ]:
if os.path.isdir("results_MobComp_A04_modified_Station10only"):
    os.chdir("results_MobComp_A04_modified_Station10only")

print("Total:")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

Results for station 10 with modified access settings, followed by station 8 with the baseline settings:


In [ ]:
if os.path.isdir("results_MobComp_A04_modified_Station10only"):
    os.chdir("results_MobComp_A04_modified_Station10only")

print("Station 10 (with modification) and Station 08 (similar to all others):")
result_sta10_offer = pd.read_csv('offer_SA10_DA1_AC2_ID_140_data_.csv', skiprows=2)
result_sta10_thrp = pd.read_csv ('thrp_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta10_delay = pd.read_csv('delay_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta08_offer = pd.read_csv('offer_SA8_DA1_AC2_ID_112_data_.csv', skiprows=2)
result_sta08_thrp = pd.read_csv ('thrp_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)
result_sta08_delay = pd.read_csv('delay_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)

plot_thrp_offer_delay(result_sta10_offer, result_sta10_thrp, result_sta10_delay,2)
plot_thrp_offer_delay(result_sta08_offer, result_sta08_thrp, result_sta08_delay,2)

os.chdir(ROOT_DIR)

### 6.6 Reduce station 10's PHY rate and increase its load

The PHY rate determines how quickly bits are transmitted while a station occupies the channel. A lower rate requires more airtime for the same amount of data. This experiment examines how that airtime use affects other senders.

In [MobComp_assignment04_modified_phy.xml](jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_phy.xml), station 10 uses 6 Mb/s instead of 54 Mb/s and offers 4 Mb/s instead of 0.75 Mb/s. All stations use AIFSN = 4, CWmin = 5, and CWmax = 100. The larger CWmax is another difference from the baseline in section 6.2.

Compare station 10, station 8, and total network throughput. Interpret differences as the combined effects of PHY rate, offered load, and contention settings. This scenario does not isolate the effect of PHY rate alone.

Results are stored in [results_MobComp_A04_modified_phy](results_MobComp_A04_modified_phy).

#### 6.6.1 Select the modified scenario


In [ ]:
os.chdir(ROOT_DIR)
SCENARIO05="jemula802/scenarios/ETH_Course/MobComp_assignment04_modified_phy.xml"
!echo Modified scenario is $SCENARIO05

import difflib
with open(SCENARIO01, 'r') as file1:
    file1_lines = file1.readlines()
with open(SCENARIO05, 'r') as file2:
    file2_lines = file2.readlines()

diff = difflib.unified_diff(file1_lines, file2_lines, fromfile=SCENARIO01, tofile=SCENARIO05, lineterm='')

#for line in diff:
#    print(line)

#### 6.6.2 Simulate and Plot

In [ ]:
os.chdir(ROOT_DIR)
!java -Xms2G -Xmx2G -classpath "$classpath_jemula802" emulator.JE802Starter $SCENARIO05

Results for all traffic:


In [ ]:
if os.path.isdir("results_MobComp_A04_modified_phy"):
    os.chdir("results_MobComp_A04_modified_phy")

print("Total:")
result_total_offer = pd.read_csv('total_offer.csv', skiprows=2)
result_total_thrp = pd.read_csv('total_thrp.csv', skiprows=2)
result_total_delay = pd.read_csv('total_delay.csv', skiprows=2)
plot_thrp_offer_delay(result_total_offer, result_total_thrp, result_total_delay,180)
plot_delay_histogram(result_total_delay.iloc[-1,7],result_total_delay.iloc[-1,8])

os.chdir(ROOT_DIR)

Results for station 10 with the lower PHY rate and higher offered load, followed by station 8:


In [ ]:
if os.path.isdir("results_MobComp_A04_modified_phy"):
    os.chdir("results_MobComp_A04_modified_phy")

print("Station 10 (with modification) and Station 08 (similar to all others):")
result_sta10_offer = pd.read_csv('offer_SA10_DA1_AC2_ID_140_data_.csv', skiprows=2)
result_sta10_thrp = pd.read_csv ('thrp_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta10_delay = pd.read_csv('delay_SA10_DA1_AC2_ID_140_data__End.csv', skiprows=2)
result_sta08_offer = pd.read_csv('offer_SA8_DA1_AC2_ID_112_data_.csv', skiprows=2)
result_sta08_thrp = pd.read_csv ('thrp_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)
result_sta08_delay = pd.read_csv('delay_SA8_DA1_AC2_ID_112_data__End.csv', skiprows=2)

plot_thrp_offer_delay(result_sta10_offer, result_sta10_thrp, result_sta10_delay,5)
plot_thrp_offer_delay(result_sta08_offer, result_sta08_thrp, result_sta08_delay,5)

os.chdir(ROOT_DIR)

----
### 6.7 Hands-On Exercises

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>


**In the baseline experiment in section 6.2, how does throughput change as offered load approaches and exceeds about 20 Mb/s? Explain the observed behavior using contention, waiting, and retransmissions.**

Answer:


**In section 6.5, does station 10 gain a throughput or delay advantage over station 8? Explain the result using their channel-access settings.**

Answer:


**In section 6.6, how do total throughput and average delay differ from the baseline in section 6.2? Consider the changes in PHY rate, offered load, and CWmax.**

Answer:


**Would a higher PHY rate for station 10 guarantee higher throughput if the other stations used lower rates? Explain how shared airtime affects the answer.**

Answer:


**OPTIONAL: Are all explanations and exercises in this notebook clear? Which parts, if any, need clarification or improvement?**

Answer:
